# Learning rate schedules

A fixed learning rate is a compromise. Early in training you want big steps to make progress. Late in training you want small ones to settle into a minimum instead of bouncing around it. A schedule changes the rate as training runs. This notebook writes the common schedules from scratch, checks one against Keras, and shows where each helps.

Order: intuition, maths, cost function, from-scratch version, library version, tuning, what breaks.

In [ ]:
from common.dl import set_seed, setup_keras

setup_keras("torch")
import keras
import matplotlib.pyplot as plt
import numpy as np

from common.datasets import load_mnist

SEED = 42
set_seed(SEED)
keras.utils.set_random_seed(SEED)
np.set_printoptions(precision=4, suppress=True)
plt.rcParams["figure.figsize"] = (6, 3.5)

## 1. Intuition

Think of parking a car. You drive fast down the street and slow to a crawl at the kerb. Training does the same. A high rate crosses the loss surface quickly. A low rate at the end lets the weights settle in the bottom of the valley instead of jittering across it.

The opposite problem sits at the start. The first few updates use random weights and noisy gradients, and a large step there can wreck the network. A short **warmup** ramps the rate up from near zero to protect against that.

## 2. Maths and cost function

With base rate $\eta_0$, step $t$ and total steps $T$:

$$\text{step: }\eta_t=\eta_0\,\gamma^{\lfloor t/s\rfloor}\qquad \text{exponential: }\eta_t=\eta_0\,\gamma^{t/s}\qquad \text{cosine: }\eta_t=\tfrac12\eta_0\big(1+\cos(\pi t/T)\big)$$

Warmup is a linear ramp for the first $w$ steps, $\eta_t=\eta_0\,t/w$, followed by any decay above. **One-cycle** ramps up to a peak for about the first third of training and then anneals down to near zero, so the rate rises and falls once.

The schedule does not change the cost function. It changes how fast the optimiser moves across it.

## 3. From scratch

Each schedule is a plain function of the step. We train for 400 steps (10 epochs of 40 batches).

In [ ]:
T = 400


def step_decay(t, lr0=0.01, gamma=0.5, every=100):
    return lr0 * gamma ** (t // every)


def exp_decay(t, lr0=0.01, gamma=0.7, every=40):
    return lr0 * gamma ** (t / every)


def cosine(t, lr0=0.01, total=T):
    return 0.5 * lr0 * (1 + np.cos(np.pi * min(t, total) / total))


def warmup_cosine(t, peak=0.01, warmup=40, total=T):
    if t < warmup:
        return peak * t / warmup
    return cosine(t - warmup, peak, total - warmup)


def one_cycle(t, peak=0.01, total=T, up=0.3, start_div=25):
    top = int(up * total)
    if t < top:
        return peak / start_div + (peak - peak / start_div) * t / top
    return 0.5 * peak * (1 + np.cos(np.pi * (t - top) / (total - top)))


steps = np.arange(T)
curves = {
    "step": step_decay,
    "exponential": exp_decay,
    "cosine": cosine,
    "warmup + cosine": warmup_cosine,
    "one-cycle": one_cycle,
}
for name, fn in curves.items():
    plt.plot(steps, [fn(t) for t in steps], label=name)
plt.legend()
plt.xlabel("step")
plt.ylabel("learning rate")
plt.show()

Step decay drops in plateaus, exponential decay is its smooth cousin, cosine falls slowly at first and quickly in the middle, and both warmup and one-cycle start low. Check the cosine against Keras to make sure the maths is right.

In [ ]:
keras_cos = keras.optimizers.schedules.CosineDecay(0.01, T)
diff = max(abs(float(keras_cos(t)) - cosine(t)) for t in steps)
print(f"max difference from Keras CosineDecay over {T} steps: {diff:.1e}")

## 4. Library version

Keras schedules are objects you pass as the `learning_rate`. `ExponentialDecay` and `CosineDecay` are built in. `CosineDecay` takes `warmup_target` and `warmup_steps` for a warmup. One-cycle is not built in, so it goes through a `LearningRateScheduler` callback that sets the rate once per epoch. The model is a two-hidden-layer MNIST net with Adam.

In [ ]:
x_tr, y_tr, x_te, y_te = load_mnist(5000, 1000, seed=SEED)
x_tr, x_te = x_tr.reshape(len(x_tr), -1), x_te.reshape(len(x_te), -1)


def make_net(lr) -> keras.Model:
    set_seed(SEED)
    model = keras.Sequential(
        [
            keras.Input((784,)),
            keras.layers.Dense(128, activation="relu"),
            keras.layers.Dense(64, activation="relu"),
            keras.layers.Dense(10, activation="softmax"),
        ]
    )
    model.compile(
        optimizer=keras.optimizers.Adam(lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


schedules = {
    "constant": (0.01, []),
    "exponential": (keras.optimizers.schedules.ExponentialDecay(0.01, 40, 0.7), []),
    "cosine": (keras.optimizers.schedules.CosineDecay(0.01, T), []),
    "warmup + cosine": (
        keras.optimizers.schedules.CosineDecay(0.0, T, warmup_target=0.01, warmup_steps=40),
        [],
    ),
    "one-cycle": (
        0.01,
        [keras.callbacks.LearningRateScheduler(lambda epoch, lr: one_cycle(epoch * 40, total=T))],
    ),
}
for name, (lr, callbacks) in schedules.items():
    model = make_net(lr)
    hist = model.fit(x_tr, y_tr, epochs=10, batch_size=128, verbose=0, callbacks=callbacks)
    acc = model.evaluate(x_te, y_te, verbose=0)[1]
    print(f"{name:>16}: final train loss {hist.history['loss'][-1]:.4f}, test acc {acc:.3f}")

Read the training loss, not the test accuracy. Cosine ends at about a third of the constant-rate loss (0.011 against 0.037), and warmup plus cosine at about an eighth (0.005). Exponential decay with this gamma does no better than constant (0.040). The test accuracies sit within half a point of each other, which is inside the noise of a 1,000-image test set. A schedule here buys a better fit, and on small data a better fit does not always buy a better test score.

## 5. Tuning

The peak rate and warmup length are the knobs. Hold the peak at 0.05, which is high for Adam, and vary the warmup.

In [ ]:
for warmup in (0, 20, 80, 160):
    sched = keras.optimizers.schedules.CosineDecay(
        0.05 if warmup == 0 else 0.0, T, warmup_target=0.05 if warmup else None, warmup_steps=warmup
    )
    model = make_net(sched)
    hist = model.fit(x_tr, y_tr, epochs=10, batch_size=128, verbose=0)
    acc = model.evaluate(x_te, y_te, verbose=0)[1]
    print(
        f"warmup {warmup:>3} steps: final train loss {hist.history['loss'][-1]:.3f}, test acc {acc:.3f}"
    )

With no warmup the loss ends at 0.145. A 20-step warmup, 5% of training, brings it to 0.010. At 160 steps it is back up to 0.119, because 40% of training is spent ramping. A warmup of 5% to 10% of the steps is a common default.

## 6. What breaks

A peak rate that Adam cannot survive from a cold start. Constant 0.1 against warmup plus cosine to the same peak, with an 80-step ramp.

In [ ]:
for name, lr in [
    ("constant 0.1", 0.1),
    (
        "warmup + cosine to 0.1",
        keras.optimizers.schedules.CosineDecay(0.0, T, warmup_target=0.1, warmup_steps=80),
    ),
]:
    model = make_net(lr)
    hist = model.fit(x_tr, y_tr, epochs=10, batch_size=128, verbose=0)
    print(
        f"{name:>24}: final train loss {hist.history['loss'][-1]:.2f}, test acc {model.evaluate(x_te, y_te, verbose=0)[1]:.3f}"
    )

The constant rate leaves the network stuck: the loss is 14.5, far above the 2.3 of a random guess, and accuracy is at chance (10%). The same peak with warmup and decay reaches about 90%. Schedules widen the range of rates that work. They do not remove the need to pick the peak sensibly.